# Contexte de marché : VIX et récessions

*Brouillon de Benjamin. Personne d'autre ne modifie ce notebook.*

**À livrer** : la fonction `charger_marche()` dans `src/donnees_marche.py`

Quand un bout de code marche ici, on le déplace dans une fonction du fichier `src/` correspondant.

In [23]:

import pandas as pd
import matplotlib.pyplot as plt

## 1. VIX

In [24]:
url = "https://r2.datahub.io/clt98ke170001ia08btn3tzgm/main/raw/data/vix-daily.csv"
vix = pd.read_csv(url)
print(vix.head())
vix.info()
vix.columns




         DATE   OPEN   HIGH    LOW  CLOSE
0  1990-01-02  17.24  17.24  17.24  17.24
1  1990-01-03  18.19  18.19  18.19  18.19
2  1990-01-04  19.22  19.22  19.22  19.22
3  1990-01-05  20.11  20.11  20.11  20.11
4  1990-01-08  20.26  20.26  20.26  20.26
<class 'pandas.DataFrame'>
RangeIndex: 9278 entries, 0 to 9277
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   DATE    9278 non-null   str    
 1   OPEN    9278 non-null   float64
 2   HIGH    9278 non-null   float64
 3   LOW     9278 non-null   float64
 4   CLOSE   9278 non-null   float64
dtypes: float64(4), str(1)
memory usage: 453.2 KB


Index(['DATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE'], dtype='str')

In [25]:
vix["DATE"] = pd.to_datetime(vix["DATE"])
vix.info()
# les dates sont désormais en mode date américaine et non string


<class 'pandas.DataFrame'>
RangeIndex: 9278 entries, 0 to 9277
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   DATE    9278 non-null   datetime64[us]
 1   OPEN    9278 non-null   float64       
 2   HIGH    9278 non-null   float64       
 3   LOW     9278 non-null   float64       
 4   CLOSE   9278 non-null   float64       
dtypes: datetime64[us](1), float64(4)
memory usage: 362.6 KB


In [26]:
print(vix["DATE"].duplicated().sum())
print(vix.isna().sum())
print(vix["DATE"].max())
# vérification de potentiels doublons, valeurs manquantes, et si l'ordre chronoligique est vérifié 

0
DATE     0
OPEN     0
HIGH     0
LOW      0
CLOSE    0
dtype: int64
2026-09-22 00:00:00


In [27]:
vix.head()
vix = vix.rename(columns={"CLOSE": "vix"})
vix.head()

,DATE,OPEN,HIGH,LOW,vix
0,1990-01-02,17.24,17.24,17.24,17.24
1,1990-01-03,18.19,18.19,18.19,18.19
2,1990-01-04,19.22,19.22,19.22,19.22
3,1990-01-05,20.11,20.11,20.11,20.11
4,1990-01-08,20.26,20.26,20.26,20.26


In [28]:
# to then facilitate coordination between the VIX table and the recessions one, we fix the date as index
vix = vix.set_index("DATE")

## 2. Récessions : scraping de la page du NBER

In [29]:
import requests
import pandas as pd
import io
%pip install lxml


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
response = requests.get("https://www.nber.org/research/data/us-business-cycle-expansions-and-contractions")
print(response.status_code)
html = response.text
dfs = pd.read_html(io.StringIO(html))
récession = dfs[0]
récession.head()
#étape de scraping



200


,0,1,2,3,4,5
0,Peak Month (Peak Quarter),Trough Month (Trough Quarter),Contraction,Expansion,Cycle,Cycle
1,Red indicates that the turning point quarter d...,Red indicates that the turning point quarter d...,"Duration, peak to trough","Duration, trough to peak","Duration, trough to trough","Duration, peak to peak"
2,NaN,December 1854 (1854Q4),NaN,NaN,NaN,NaN
3,June 1857 (1857Q2),December 1858 (1858Q4),18,30,48,NaN
4,October 1860 (1860Q3),June 1861 (1861Q3),8,22,30,40


In [31]:
print(récession.tail(10))
print(récession.shape)
print(récession.columns)
print(récession.info())
#statistiques descriptives afin de déterminer les informations inutiles

                         0                       1     2     3     4     5
32      July 1981 (1981Q3)  November 1982 (1982Q4)    16    12    28    18
33      July 1990 (1990Q3)     March 1991 (1991Q1)     8    92   100   108
34     March 2001 (2001Q1)  November 2001 (2001Q4)     8   120   128   128
35  December 2007 (2007Q4)      June 2009 (2009Q2)    18    73    91    81
36  February 2020 (2019Q4)     April 2020 (2020Q2)     2   128   130   146
37                     NaN                     NaN   NaN   NaN   NaN   NaN
38               1854-2020               1854-2020  17.0  41.4  58.4  59.2
39               1854-1919               1854-1919  21.6  26.6  48.2  48.9
40               1919-1945               1919-1945  18.2  35.0  53.2  53.0
41               1945-2020               1945-2020  10.3  64.2  74.5  75.0
(42, 6)
Index([0, 1, 2, 3, 4, 5], dtype='int64')
<class 'pandas.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype


In [32]:
récession = récession.iloc[2:-5]
récession = récession[[0,1]]




In [33]:
récession.columns = ["start", "end"]
récession.head()


,start,end
2,NaN,December 1854 (1854Q4)
3,June 1857 (1857Q2),December 1858 (1858Q4)
4,October 1860 (1860Q3),June 1861 (1861Q3)
5,April 1865 (1865Q1),December 1867 (1868Q1)
6,June 1869 (1869Q2),December 1870 (1870Q4)


In [34]:
récession[["start", "end"]] = récession[["start", "end"]].apply(lambda x : x.str.strip())
récession[["start", "end"]]= récession[["start", "end"]].apply(lambda x : x.str.split("(").str[0].str.strip())
récession[["start", "end"]] = récession[["start","end"]].apply(pd.to_datetime, format = "%B %Y")
récession.head()


,start,end
2,NaT,1854-12-01
3,1857-06-01,1858-12-01
4,1860-10-01,1861-06-01
5,1865-04-01,1867-12-01
6,1869-06-01,1870-12-01


In [35]:
récession = récession.dropna(subset = ["start", "end"])
récession.head()

,start,end
3,1857-06-01,1858-12-01
4,1860-10-01,1861-06-01
5,1865-04-01,1867-12-01
6,1869-06-01,1870-12-01
7,1873-10-01,1879-03-01


## 3. Table quotidienne et niveaux de stress

In [47]:
# we create a new column in our vix table. It will take 0 as a value when we are out of recession, against one when there is a recession
vix["récession"] = 0

for i in range(len(récession)):
    debut = récession["start"].iloc[i] + pd.DateOffset(months=1) #indeed we follow the convention that recession starts one month after NBER peak
    fin = récession["end"].iloc[i] + pd.offsets.MonthEnd(0)  #it's a second convention : récession ends at the end of the trough month (i.e le mois de creux)

for date in vix.index:
        if date >= debut and date <= fin:
            vix.loc[date, "récession"] = 1


In [48]:
#On vient créer une autre colonne pour nous dire dans quel tiers la valeur du vixe à chaque ligne appartient
vix["stress"] = pd.qcut(vix["vix"], q = 3, labels = ["bas", "moyen", "haut" ])


In [52]:
# Finally, I prepare the final format so that Diego can easily use my code and table 
marché = vix.reset_index() # needless for the date to be an index anymore
marché = marché.rename(columns = {"DATE" : "date"})
marché = marché[["date", "vix", "récession", "stress"]]
marché.tail(40)

,date,vix,récession,stress
9238,2026-07-29,20.66,0,moyen
9239,2026-07-30,17.09,0,moyen
9240,2026-07-31,15.99,0,moyen
9241,2026-08-03,15.86,0,moyen
9242,2026-08-04,16.50,0,moyen
9243,2026-08-05,15.81,0,moyen
9244,2026-08-06,15.15,0,bas
9245,2026-08-07,14.90,0,bas
9246,2026-08-10,15.46,0,moyen
9247,2026-08-11,15.28,0,moyen
